In [ ]:
!pip install geopandas folium xgboost statsmodels

In [4]:
import datetime
import logging
import time
from typing import Any, Dict, List, Optional
import pandas as pd
import requests
import numpy as np

## Data Loading 

In [ ]:
logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)


class USGSDataPipeline:
    """A robust data pipeline to fetch, clean, and structure historical earthquake

    data from the USGS API over extended timeframes.
    """

    BASE_URL = "https://earthquake.usgs.gov/fdsnws/event/1/query"

    def __init__(
        self,
        min_lat: float,
        max_lat: float,
        min_lon: float,
        max_lon: float,
        min_magnitude: float = 2.5,
    ):
        self.bounds = {
            "minlatitude": min_lat,
            "maxlatitude": max_lat,
            "minlongitude": min_lon,
            "maxlongitude": max_lon,
            "minmagnitude": min_magnitude,
            "format": "geojson",
        }

    def fetch_yearly_chunk(
        self, start_year: int, end_year: int
    ) -> List[Dict[str, Any]]:
        """Fetches earthquake data in yearly intervals to bypass the USGS 20,000

        record limit.
        """
        all_features = []

        for year in range(start_year, end_year + 1):
            params = self.bounds.copy()
            params["starttime"] = f"{year}-01-01"
            params["endtime"] = f"{year}-12-31T23:59:59"

            logger.info(
                f"Fetching data for year {year} from magnitude {params['minmagnitude']}..."
            )

            try:
                response = requests.get(
                    self.BASE_URL, params=params, timeout=30
                )
                response.raise_for_status()
                data = response.json()

                features = data.get("features", [])
                logger.info(f"Successfully retrieved {len(features)} events.")
                all_features.extend(features)

                # Politeness delay to prevent rate-limiting
                time.sleep(1)

            except requests.exceptions.RequestException as e:
                logger.error(
                    f"Failed to fetch data for year {year}. Error: {e}"
                )
                continue

        return all_features

    def transform_geojson_to_df(
        self, features: List[Dict[str, Any]]
    ) -> pd.DataFrame:
        """Flattens the nested GeoJSON structure into a clean Pandas DataFrame

        and normalizes data types.
        """
        if not features:
            logger.warning("No features provided for transformation.")
            return pd.DataFrame()

        extracted_data = []

        for feature in features:
            props = feature.get("properties", {})
            geom = feature.get("geometry", {})
            coords = geom.get("coordinates", [None, None, None])

            # Extract coordinates safely (Longitude, Latitude, Depth)
            lon = coords[0] if len(coords) > 0 else None
            lat = coords[1] if len(coords) > 1 else None
            depth = coords[2] if len(coords) > 2 else None

            event = {
                "event_id": feature.get("id"),
                "time_epoch": props.get("time"),  # Milliseconds since epoch
                "magnitude": props.get("mag"),
                "place": props.get("place"),
                "latitude": lat,
                "longitude": lon,
                "depth_km": depth,
                "updated_epoch": props.get("updated"),
                "alert": props.get("alert"),
                "status": props.get("status"),
            }
            extracted_data.append(event)

        df = pd.DataFrame(extracted_data)

        # Post-processing and type conversions
        if not df.empty:
            # Convert milliseconds epoch to standard UTC datetime
            df["timestamp"] = pd.to_datetime(
                df["time_epoch"], unit="ms", utc=True
            )
            # Remove rows where crucial spatial-temporal or magnitude data is missing
            df = df.dropna(subset=["timestamp", "latitude", "longitude", "magnitude"])
            # Sort chronologically
            df = df.sort_values(by="timestamp").reset_index(drop=True)

        return df

    def run_pipeline(self, start_year: int, end_year: int) -> pd.DataFrame:
        """Executes the end-to-end extraction and transformation process."""
        logger.info(
            f"Starting seismic data pipeline from {start_year} to {end_year}."
        )
        raw_features = self.fetch_yearly_chunk(start_year, end_year)
        processed_df = self.transform_geojson_to_df(raw_features)
        logger.info(
            f"Pipeline complete. Total valid records processed: {len(processed_df)}"
        )
        return processed_df


# --- EXECUTION BLOCK ---
if __name__ == "__main__":
    SO_CAL_MIN_LAT = 32.5
    SO_CAL_MAX_LAT = 37.0
    SO_CAL_MIN_LON = -121.0
    SO_CAL_MAX_LON = -114.0

    START_YR = 1996
    END_YR = 2026

    pipeline = USGSDataPipeline(
        min_lat=SO_CAL_MIN_LAT,
        max_lat=SO_CAL_MAX_LAT,
        min_lon=SO_CAL_MIN_LON,
        max_lon=SO_CAL_MAX_LON,
        min_magnitude=2.5, 
    )

    df_seismic = pipeline.run_pipeline(start_year=START_YR, end_year=END_YR)

    if not df_seismic.empty:
        output_filename = "southern_california_seismic_30yr.csv"
        df_seismic.to_csv(output_filename, index=False)
        print(f"\nSuccessfully saved dataset to {output_filename}")
        print(df_seismic.head())

## Data Preprocessing 

In [5]:
def haversine_distance(
    lat1: pd.Series, lon1: pd.Series, lat2: float, lon2: float
) -> pd.Series:
    """Calculates the great-circle distance (in km) between a moving point (Series)

    and a fixed mainshock coordinate using the Haversine formula.
    """
    R = 6371.0  

    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)

    a = (
        np.sin(dphi / 2.0) ** 2
        + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda / 2.0) ** 2
    )
    c = 2.0 * np.arctan2(np.sqrt(a), np.sqrt(1.0 - a))
    return R * c


class SeismicCatalogCleaner:

    def __init__(self, filepath: str):
        self.filepath = filepath
        self.df = pd.read_csv(filepath)

    def sanity_scrub(self) -> pd.DataFrame:
        """Phase 1: Basic physical constraints and deduplication."""
        print(f"Original raw rows loaded: {len(self.df)}")

       
        self.df["timestamp"] = pd.to_datetime(self.df["timestamp"], format="ISO8601", utc=True)

        self.df = self.df.drop_duplicates(subset=["event_id"])

        self.df = self.df[
            (self.df["depth_km"] >= -2.0) & (self.df["depth_km"] <= 100.0)
        ]

        self.df = self.df[self.df["magnitude"] >= 2.5]

        if "type" in self.df.columns:
            self.df = self.df[self.df["type"] == "earthquake"]

        self.df = self.df.sort_values(by="timestamp").reset_index(drop=True)
        print(f"Rows remaining after physical scrub: {len(self.df)}")
        return self.df
        
    def decluster_and_tag_sequences(
        self, ms_cutoff: float = 5.5, window_days: int = 60, radius_km: float = 100.0
    ) -> pd.DataFrame:
        print(f"\nTagging Aftershock sequences for Mainshocks (M >= {ms_cutoff})...")

        self.df["is_mainshock"] = False
        self.df["aftershock_of_event_id"] = None
        self.df["dt_days"] = np.nan
        self.df["dr_km"] = np.nan

        candidates = self.df[self.df["magnitude"] >= ms_cutoff].copy()

        foreshock_ids = []
        for _, ms in candidates.iterrows():
            # Look ahead 7 days from this quake
            future_window = candidates[
                (candidates["timestamp"] > ms["timestamp"])
                & (
                    candidates["timestamp"]
                    <= ms["timestamp"] + pd.Timedelta(days=7)
                )
            ]
            # If a FUTURE quake in that window is BIGGER and nearby, demote this one
            if not future_window.empty:
                dists = haversine_distance(
                    future_window["latitude"],
                    future_window["longitude"],
                    ms["latitude"],
                    ms["longitude"],
                )
                bigger_nearby = future_window[
                    (dists <= radius_km)
                    & (future_window["magnitude"] > ms["magnitude"])
                ]
                if not bigger_nearby.empty:
                    foreshock_ids.append(ms["event_id"])

        true_mainshocks = candidates[
            ~candidates["event_id"].isin(foreshock_ids)
        ]

        for _, ms in true_mainshocks.iterrows():
            ms_time = ms["timestamp"]
            ms_lat = ms["latitude"]
            ms_lon = ms["longitude"]
            ms_mag = ms["magnitude"]
            ms_id = ms["event_id"]

            self.df.loc[self.df["event_id"] == ms_id, "is_mainshock"] = True

            time_mask = (self.df["timestamp"] > ms_time) & (
                self.df["timestamp"] <= ms_time + pd.Timedelta(days=window_days)
            )
            dists = haversine_distance(
                self.df["latitude"], self.df["longitude"], ms_lat, ms_lon
            )

            aftershock_mask = (
                time_mask
                & (dists <= radius_km)
                & (self.df["magnitude"] < ms_mag)
                & (self.df["aftershock_of_event_id"].isna())
            )

            self.df.loc[aftershock_mask, "aftershock_of_event_id"] = ms_id
            self.df.loc[aftershock_mask, "dt_days"] = (
                self.df.loc[aftershock_mask, "timestamp"] - ms_time
            ).dt.total_seconds() / 86400.0
            self.df.loc[aftershock_mask, "dr_km"] = dists[aftershock_mask]

        print(f" -> Verified {len(true_mainshocks)} True Mainshocks.")
        print(
            f" -> Demoted {len(foreshock_ids)} foreshocks (including Ridgecrest M6.4)."
        )
        return self.df
    
            

    def save_clean_catalog(self, output_path: str):
        self.df.to_csv(output_path, index=False)
        print(f"\nPreprocessed catalog saved successfully to: {output_path}")


# --- EXECUTION ---
if __name__ == "__main__":
    INPUT_CSV = "southern_california_seismic_30yr.csv"
    OUTPUT_CSV = "southern_california_cleaned_declustered.csv"

    cleaner = SeismicCatalogCleaner(filepath=INPUT_CSV)
    df_scrubbed = cleaner.sanity_scrub()
    df_final = cleaner.decluster_and_tag_sequences(
        ms_cutoff=5.5, window_days=60, radius_km=100.0
    )

    cleaner.save_clean_catalog(OUTPUT_CSV)

Original raw rows loaded: 19424
Rows remaining after physical scrub: 19423

Tagging Aftershock sequences for Mainshocks (M >= 5.5)...
 -> Verified 9 True Mainshocks.
 -> Demoted 1 foreshocks (including Ridgecrest M6.4).

Preprocessed catalog saved successfully to: southern_california_cleaned_declustered.csv
